# Setup

In [101]:
import pandas as pd
import os
import datacompy
import numpy as np

# Import Data

In [102]:
df_delin_2022 = pd.read_excel("../data/census_data/excel_delineation_2022.xls",skiprows=2,index_col=None,dtype=object, engine="xlrd").dropna(subset=['CBSA Code','FIPS State Code'])
df_delin_2023 = pd.read_excel("../data/census_data/excel_delineation_2023.xlsx",skiprows=2,index_col=None,dtype=object).dropna(subset=['CBSA Code','FIPS State Code'])

In [103]:
# this census file includes includes both old and new CT county code rows
df_census_2023 = pd.read_csv("../output/ffiec_census_msamd_names_2023.txt",delimiter="|", dtype=object )

df_census_2024_raw = df_census_2023.copy()
df_census_2024_raw["Collection Year"] = "2024"
# df_census_2024_raw["COUNTY_CODE_2024"] = df_census_2024_raw["County"]
# df_census_2024_old = pd.read_csv("../output_FHFA/ffiec_census_2024_with_new_CT_county_codes.txt",delimiter="|", dtype=object )

df_CT_map = pd.read_csv("../output_FHFA/connecticut_county_code_mapping.csv", dtype=object)


In [104]:
df_ct_total = df_census_2024_raw[(df_census_2024_raw['State' ] == '09')]

In [105]:
df_CT_map.columns

Index(['NAMELSAD', 'STATEFP', 'old_county_code', 'new_county_code'], dtype='object')

In [106]:
# df_2024_T = (df_census_2024_raw.merge(df2, on=cols_to_be_matched, how='left', suffixes=('','\x00'))
#        .sort_index(axis=1).bfill(axis=1)[df.columns])


In [107]:
# df_ct_total = df_census_2023[(df_census_2023['State' ] == '09')]

# df_ct_old = df_census_2023[(df_census_2023['State' ] == '09') & (df_census_2023['County'].astype(int) <= 15)]
# df_ct_new = df_census_2024_old[(df_census_2024_old['State' ] == '09') & (df_census_2024_old['County'].astype(int) > 15)]

# df_census_2023_no_CT = df_census_2023[(df_census_2023['State' ] != '09')]

In [108]:
print(df_delin_2023.columns)
delin_keep_cols = ['CBSA Code', 'CBSA Title','FIPS State Code', 'FIPS County Code' ]
df_delin_2022 = df_delin_2022[delin_keep_cols]
df_delin_2023 = df_delin_2023[delin_keep_cols]

Index(['CBSA Code', 'Metropolitan Division Code', 'CSA Code', 'CBSA Title',
       'Metropolitan/Micropolitan Statistical Area',
       'Metropolitan Division Title', 'CSA Title', 'County/County Equivalent',
       'State Name', 'FIPS State Code', 'FIPS County Code',
       'Central/Outlying County'],
      dtype='object')


In [109]:


# df_delin_2023 = df_delin_2023.merge(df_CT_map,  how = "left", left_on = ['FIPS State Code', 'FIPS County Code'],right_on = ['STATEFP', 'new_county_code'],suffixes=('_2023',""))


In [110]:
df_delin_2022 = df_delin_2022.merge(df_CT_map, how = "left", left_on = ['FIPS State Code', 'FIPS County Code'],right_on = ['STATEFP', 'old_county_code'],suffixes=('_2022',""))

df_delin_2023 = df_delin_2023.merge(df_CT_map,  how = "left", left_on = ['FIPS State Code', 'FIPS County Code'],right_on = ['STATEFP', 'new_county_code'],suffixes=('_2023',""))


df_delin_2022["COUNTY_CODE"] = df_delin_2022["new_county_code"].fillna(df_delin_2023['FIPS County Code'] )
df_delin_2022["STATE"] = df_delin_2022['FIPS State Code']
df_delin_2022["MSA_NAME"] = df_delin_2022['CBSA Title']
df_delin_2022["MSA_ID"] = df_delin_2022['CBSA Code'].str.zfill(6)
df_delin_2022["TRACT"] = df_delin_2022["NAMELSAD"]

df_delin_2022 = df_delin_2022.drop_duplicates()

df_delin_2023["COUNTY_CODE"] = df_delin_2023["new_county_code"].fillna(df_delin_2023['FIPS County Code'] )
df_delin_2023["STATE"] = df_delin_2023['FIPS State Code']
df_delin_2023["MSA_NAME"] = df_delin_2023['CBSA Title']
df_delin_2023["MSA_ID"] = df_delin_2023['CBSA Code'].str.zfill(6)
df_delin_2023["TRACT"] = df_delin_2023["NAMELSAD"]





In [83]:
print(df_delin_2022.columns)
print()
print(df_delin_2023.columns)

Index(['CBSA Code', 'CBSA Title', 'FIPS State Code', 'FIPS County Code',
       'NAMELSAD', 'STATEFP', 'old_county_code', 'new_county_code',
       'COUNTY_CODE', 'STATE', 'MSA_NAME', 'MSA_ID', 'TRACT'],
      dtype='object')

Index(['CBSA Code', 'CBSA Title', 'FIPS State Code', 'FIPS County Code',
       'NAMELSAD', 'STATEFP', 'old_county_code', 'new_county_code',
       'COUNTY_CODE', 'STATE', 'MSA_NAME', 'MSA_ID', 'TRACT'],
      dtype='object')


In [84]:
df_delin_2022 = df_delin_2022[['COUNTY_CODE', 'STATE', 'MSA_NAME', 'MSA_ID']]
df_delin_2023 = df_delin_2023[["old_county_code",'COUNTY_CODE', 'STATE', 'MSA_NAME', 'MSA_ID']]

In [85]:
print(df_delin_2022.columns)
print()
print(df_delin_2023.columns)

Index(['COUNTY_CODE', 'STATE', 'MSA_NAME', 'MSA_ID'], dtype='object')

Index(['old_county_code', 'COUNTY_CODE', 'STATE', 'MSA_NAME', 'MSA_ID'], dtype='object')


# TEST

In [111]:
df_census_2024_raw.columns


Index(['Collection Year', 'MSA/MD', 'State', 'County', 'Census Tract',
       'FFIEC Median Family Income', 'Population', 'Minority Population %',
       'Number of Owner Occupied Units', 'Number of 1 to 4 Family Units',
       'Tract MFI', 'Tract to MSA Income %', 'Median Age', 'Small County',
       'MSA/MD Name', 'COUNTY_CODE_2024'],
      dtype='object')

In [127]:
df_delin_2023.columns
df_delin_2023_t = df_delin_2023.rename(columns = {
    "STATEFP" : "State",
    "COUNTY_CODE" : "County" ,
    "MSA_NAME" : 'MSA/MD Name'
})
df_delin_2023_t.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2795 entries, 0 to 2794
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   CBSA Code         2795 non-null   object
 1   CBSA Title        2795 non-null   object
 2   FIPS State Code   2795 non-null   object
 3   FIPS County Code  2795 non-null   object
 4   NAMELSAD          889 non-null    object
 5   State             889 non-null    object
 6   old_county_code   889 non-null    object
 7   new_county_code   889 non-null    object
 8   County            2795 non-null   object
 9   STATE             2795 non-null   object
 10  MSA/MD Name       2795 non-null   object
 11  MSA_ID            2795 non-null   object
 12  TRACT             889 non-null    object
dtypes: object(13)
memory usage: 370.2+ KB


In [128]:
# df_2024_T = df_census_2024_raw.mask(df_census_2024_raw["COUNTY_CODE_2024"] == "09" , df_delin_2023)
# df1 = (df_census_2024_raw.merge(df_delin_2023, on=cols_to_be_matched, how='left', suffixes=('','\x00'))
#        .sort_index(axis=1).bfill(axis=1)[df.columns])
df1 = df_census_2024_raw.set_index(['State', 'County','MSA/MD Name']).fillna(df_delin_2023_t.set_index(['State', 'County', 'MSA/MD Name'])).reset_index()


In [ ]:

df_delin_all = df_delin_2022.merge(df_delin_2023, on = ['COUNTY_CODE', 'STATE'],indicator=True , suffixes=('_2022',"_2023"))
# ffiec_census_df.loc[(ffiec_census_df['MSA/MD'] == "99999"), 'MSA/MD Name'] = "" # added 2/8/24

df_delin_all.columns

In [86]:
df_census_2024 = df_census_2024_raw.merge(df_delin_2023, left_on = ['State', 'County'],right_on = ['STATE', 'old_county_code'],indicator=True )
# ffiec_census_df.loc[(ffiec_census_df['MSA/MD'] == "99999"), 'MSA/MD Name'] = "" # added 2/8/24

df_census_2024.columns


Index(['Collection Year', 'MSA/MD', 'State', 'County', 'Census Tract',
       'FFIEC Median Family Income', 'Population', 'Minority Population %',
       'Number of Owner Occupied Units', 'Number of 1 to 4 Family Units',
       'Tract MFI', 'Tract to MSA Income %', 'Median Age', 'Small County',
       'MSA/MD Name', 'COUNTY_CODE_2024', 'old_county_code', 'COUNTY_CODE',
       'STATE', 'MSA_NAME', 'MSA_ID', '_merge'],
      dtype='object')

In [ ]:
# df_ct_total = df_census_2023[(df_census_2023['State' ] == '09')]

# df_ct_old = df_census_2023[(df_census_2023['State' ] == '09') & (df_census_2023['County'].astype(int) <= 15)]
# df_ct_new = df_census_2024_old[(df_census_2024_old['State' ] == '09') & (df_census_2024_old['County'].astype(int) > 15)]

# df_census_2023_no_CT = df_census_2023[(df_census_2023['State' ] != '09')]

In [ ]:
df_census_2024.head()

# CT update

In [ ]:
df_ct_total = df_census_2024[(df_census_2024['State' ] == '09')]

df_ct_old = df_census_2024[(df_census_2024['State' ] == '09') & (df_census_2024['County'].astype(int) <= 15)]
df_ct_new = df_census_2024[(df_census_2024['State' ] == '09') & (df_census_2024['County'].astype(int) > 15)]

df_census_2024_no_CT = df_census_2024[(df_census_2024['State' ] != '09')]



In [ ]:
df_2023_compare2 = datacompy.Compare(df_ct_old,df_ct_new, df1_name = "old", df2_name= "new",join_columns=["Census Tract", "State"])
print(df_2023_compare2.report())

In [ ]:
df_2023_all_mismatch = df_2023_compare2.all_mismatch()
# df_2023_all_mismatch = df_2023_all_mismatch[['collection year_df1', 'msa/md_df1','msa/md name_df1',
#        'msa/md_df2', 'msa/md name_df2', 'census tract_df1', 'census tract_df2','state_df1', 'state_df2', 'county_df1', 'county_df2']]
df_2023_all_mismatch = df_2023_all_mismatch[['collection year_df1', 'msa/md_df1','msa/md name_df1',
       'msa/md_df2', 'msa/md name_df2', 'census tract','state', 'county_df1', 'county_df2']]


# Update Collection Year


In [ ]:
df_census_2024_with_new_CT_county_codes = pd.concat([df_census_2024_no_CT, df_ct_new])
df_census_2024_with_new_CT_county_codes['Collection Year'] = 2024


# check counts

In [ ]:
df_census_2024_no_CT.shape[0] + df_ct_new.shape[0] == df_census_2024_with_new_CT_county_codes.shape[0]

In [ ]:
# if 'MSA/MD' is 99999 then 'MSA/MD Name' is "Not applicable"
# df_census_2024_with_new_CT_county_codes.loc[(df_census_2024_with_new_CT_county_codes['MSA/MD'] == 99999), 'MSA/MD Name'] = np.NaN

In [ ]:
# df_census_2024_with_new_CT_county_codes[(df_census_2024_with_new_CT_county_codes['MSA/MD'] == 99999)].count()


In [ ]:
df_census_2024_with_new_CT_county_codes[(df_census_2024_with_new_CT_county_codes['MSA/MD'] != 99999)].count()

# Write Output

In [ ]:
df_census_2024_with_new_CT_county_codes.to_csv("../output_FHFA/" + "ffiec_census_{year}_with_new_CT_county_codes.{end}".format(year="2024", end="txt"), 
								   index=False, 
								   sep="|")

